# Historical Fight Weighting

This notebook tests whether reducing the influence of older fights improves validation performance. The test set remains excluded from every comparison.

## 1. Initial Strategy Comparison

Compare no weighting, a hard cutoff, step weighting, and exponential decay.


In [1]:
from pathlib import Path
import sys
import pandas as pd

current = Path.cwd().resolve()
project_root = next(
    p for p in [current, *current.parents]
    if (p / "data/processed/ufcstats/fight_dataset.parquet").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.modeling.time_weighted_training import (
    TimeWeightConfig,
    train_time_weighted_experiment,
)
from src.modeling.ufc_modeling import (
    RandomForestConfig, CatBoostConfig, SplitConfig
)

In [2]:
rf_config = RandomForestConfig()
cat_config = CatBoostConfig()
split_config = SplitConfig()

strategies = {
    "none": TimeWeightConfig(mode="none"),
    "exclude_before_2005": TimeWeightConfig(
        mode="exclude_before", cutoff_date="2005-01-01"
    ),
    "step_before_2005_025": TimeWeightConfig(
        mode="step", cutoff_date="2005-01-01", pre_cutoff_weight=0.25
    ),
    "exponential_half_life_8": TimeWeightConfig(
        mode="exponential", half_life_years=8, minimum_weight=0.10
    ),
}

In [3]:
results = {}
for name, config in strategies.items():
    print("\n" + "=" * 80)
    print(name)
    results[name] = train_time_weighted_experiment(
        project_root=project_root,
        time_weight_config=config,
        rf_config=rf_config,
        cat_config=cat_config,
        split_config=split_config,
        evaluate_test=False,
        experiment_name=name,
    )


none
Temporal weighting:
  mode=none; original train=6,307; effective train=6,307; mean weight=1.0000; effective sample size=6307.0
Training Random Forest...
Training CatBoost...
0:	learn: 0.6919777	test: 0.6921598	best: 0.6921598 (0)	total: 119ms	remaining: 3m 57s
100:	learn: 0.6436785	test: 0.6693868	best: 0.6693868 (100)	total: 5.23s	remaining: 1m 38s
200:	learn: 0.6269106	test: 0.6643399	best: 0.6643151 (198)	total: 10.1s	remaining: 1m 30s
300:	learn: 0.6131275	test: 0.6635690	best: 0.6632978 (288)	total: 15s	remaining: 1m 24s
400:	learn: 0.5961288	test: 0.6627661	best: 0.6627661 (400)	total: 19.9s	remaining: 1m 19s
500:	learn: 0.5796968	test: 0.6623928	best: 0.6620925 (493)	total: 24.8s	remaining: 1m 14s
600:	learn: 0.5633403	test: 0.6636993	best: 0.6619903 (538)	total: 29.7s	remaining: 1m 9s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.66199032
bestIteration = 538

Shrink model to first 539 iterations.

Training complete.
Selected model: catboost
     spl

In [4]:
rows = []
for strategy, result in results.items():
    for model, metrics in result["validation_metrics"].items():
        rows.append({
            "strategy": strategy,
            "model": model,
            **metrics,
            "effective_train_rows": result["weighting_summary"]["effective_train_rows"],
            "effective_sample_size": result["weighting_summary"]["effective_sample_size"],
            "mean_weight": result["weighting_summary"]["weight_mean"],
        })

comparison = pd.DataFrame(rows).sort_values(
    ["log_loss", "brier_score", "roc_auc"],
    ascending=[True, True, False],
)
comparison

,strategy,model,accuracy,balanced_accuracy,roc_auc,log_loss,brier_score,threshold,tn,fp,fn,tp,effective_train_rows,effective_sample_size,mean_weight
3,exclude_before_2005,catboost,0.596040,0.597556,0.645250,0.659479,0.233786,0.5,292,232,176,310,5877,5877.000000,1.000000
5,step_before_2005_025,catboost,0.598020,0.599763,0.645937,0.659559,0.233761,0.5,290,234,172,314,6307,6066.226038,0.948866
1,none,catboost,0.593069,0.594395,0.639753,0.661990,0.234977,0.5,293,231,180,306,6307,6307.000000,1.000000
4,step_before_2005_025,random_forest,0.604950,0.605995,0.646393,0.662721,0.235167,0.5,303,221,178,308,6307,6066.226038,0.948866
0,none,random_forest,0.612871,0.614076,0.646797,0.662859,0.235200,0.5,305,219,172,314,6307,6307.000000,1.000000
2,exclude_before_2005,random_forest,0.601980,0.602759,0.644178,0.663330,0.235441,0.5,305,219,183,303,5877,5877.000000,1.000000
6,exponential_half_life_8,random_forest,0.604950,0.605622,0.641418,0.663668,0.235625,0.5,308,216,183,303,6307,5423.037173,0.571970
7,exponential_half_life_8,catboost,0.600990,0.602775,0.635276,0.664476,0.236062,0.5,291,233,170,316,6307,5423.037173,0.571970


## 2. Broader Cutoff Search

The initial comparison suggests that a hard cutoff may help CatBoost, so more recent cutoff dates are tested before selecting a strategy.


In [5]:
rf_config = RandomForestConfig()
cat_config = CatBoostConfig()
split_config = SplitConfig()

strategies = {
    "none": TimeWeightConfig(mode="none"),
    "exclude_before_2010": TimeWeightConfig(
        mode="exclude_before", cutoff_date="2010-01-01"
    ),
    "step_before_2010_025": TimeWeightConfig(
        mode="step", cutoff_date="2010-01-01", pre_cutoff_weight=0.25
    ),
    "exponential_half_life_5": TimeWeightConfig(
        mode="exponential", half_life_years=5, minimum_weight=0.10
    ),
}

In [6]:
results = {}
for name, config in strategies.items():
    print("\n" + "=" * 80)
    print(name)
    results[name] = train_time_weighted_experiment(
        project_root=project_root,
        time_weight_config=config,
        rf_config=rf_config,
        cat_config=cat_config,
        split_config=split_config,
        evaluate_test=False,
        experiment_name=name,
    )


none
Temporal weighting:
  mode=none; original train=6,307; effective train=6,307; mean weight=1.0000; effective sample size=6307.0
Training Random Forest...
Training CatBoost...
0:	learn: 0.6919777	test: 0.6921598	best: 0.6921598 (0)	total: 43.3ms	remaining: 1m 26s
100:	learn: 0.6436785	test: 0.6693868	best: 0.6693868 (100)	total: 5.51s	remaining: 1m 43s
200:	learn: 0.6269106	test: 0.6643399	best: 0.6643151 (198)	total: 10.8s	remaining: 1m 36s
300:	learn: 0.6131275	test: 0.6635690	best: 0.6632978 (288)	total: 16s	remaining: 1m 30s
400:	learn: 0.5961288	test: 0.6627661	best: 0.6627661 (400)	total: 21.3s	remaining: 1m 24s
500:	learn: 0.5796968	test: 0.6623928	best: 0.6620925 (493)	total: 26.6s	remaining: 1m 19s
600:	learn: 0.5633403	test: 0.6636993	best: 0.6619903 (538)	total: 31.9s	remaining: 1m 14s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.66199032
bestIteration = 538

Shrink model to first 539 iterations.

Training complete.
Selected model: catboost
     s

In [7]:
rows = []
for strategy, result in results.items():
    for model, metrics in result["validation_metrics"].items():
        rows.append({
            "strategy": strategy,
            "model": model,
            **metrics,
            "effective_train_rows": result["weighting_summary"]["effective_train_rows"],
            "effective_sample_size": result["weighting_summary"]["effective_sample_size"],
            "mean_weight": result["weighting_summary"]["weight_mean"],
        })

comparison = pd.DataFrame(rows).sort_values(
    ["log_loss", "brier_score", "roc_auc"],
    ascending=[True, True, False],
)
comparison

,strategy,model,accuracy,balanced_accuracy,roc_auc,log_loss,brier_score,threshold,tn,fp,fn,tp,effective_train_rows,effective_sample_size,mean_weight
3,exclude_before_2010,catboost,0.599010,0.600642,0.646208,0.661049,0.234352,0.5,292,232,173,313,5057,5057.000000,1.000000
1,none,catboost,0.593069,0.594395,0.639753,0.661990,0.234977,0.5,293,231,180,306,6307,6307.000000,1.000000
4,step_before_2010_025,random_forest,0.605941,0.607023,0.647850,0.662132,0.234873,0.5,303,221,177,309,6307,5614.572235,0.851356
5,step_before_2010_025,catboost,0.598020,0.599837,0.642042,0.662412,0.235057,0.5,289,235,171,315,6307,5614.572235,0.851356
2,exclude_before_2010,random_forest,0.607921,0.609006,0.646165,0.662823,0.235191,0.5,304,220,176,310,5057,5057.000000,1.000000
0,none,random_forest,0.612871,0.614076,0.646797,0.662859,0.235200,0.5,305,219,172,314,6307,6307.000000,1.000000
7,exponential_half_life_5,catboost,0.583168,0.584928,0.638272,0.663496,0.235596,0.5,282,242,179,307,6307,4730.614526,0.445135
6,exponential_half_life_5,random_forest,0.600000,0.601224,0.638799,0.664856,0.236203,0.5,298,226,178,308,6307,4730.614526,0.445135


In [8]:
rf_config = RandomForestConfig()
cat_config = CatBoostConfig()
split_config = SplitConfig()

strategies = {
    "none": TimeWeightConfig(mode="none"),
    "exclude_before_2015": TimeWeightConfig(
        mode="exclude_before", cutoff_date="2015-01-01"
    ),
    "step_before_2015_025": TimeWeightConfig(
        mode="step", cutoff_date="2015-01-01", pre_cutoff_weight=0.25
    ),
    "exponential_half_life_8": TimeWeightConfig(
        mode="exponential", half_life_years=8, minimum_weight=0.10
    ),
}

In [9]:
results = {}
for name, config in strategies.items():
    print("\n" + "=" * 80)
    print(name)
    results[name] = train_time_weighted_experiment(
        project_root=project_root,
        time_weight_config=config,
        rf_config=rf_config,
        cat_config=cat_config,
        split_config=split_config,
        evaluate_test=False,
        experiment_name=name,
    )


none
Temporal weighting:
  mode=none; original train=6,307; effective train=6,307; mean weight=1.0000; effective sample size=6307.0
Training Random Forest...
Training CatBoost...
0:	learn: 0.6919777	test: 0.6921598	best: 0.6921598 (0)	total: 48.2ms	remaining: 1m 36s
100:	learn: 0.6436785	test: 0.6693868	best: 0.6693868 (100)	total: 5.65s	remaining: 1m 46s
200:	learn: 0.6269106	test: 0.6643399	best: 0.6643151 (198)	total: 11.1s	remaining: 1m 38s
300:	learn: 0.6131275	test: 0.6635690	best: 0.6632978 (288)	total: 16.5s	remaining: 1m 32s
400:	learn: 0.5961288	test: 0.6627661	best: 0.6627661 (400)	total: 22.1s	remaining: 1m 28s
500:	learn: 0.5796968	test: 0.6623928	best: 0.6620925 (493)	total: 27.6s	remaining: 1m 22s
600:	learn: 0.5633403	test: 0.6636993	best: 0.6619903 (538)	total: 33.1s	remaining: 1m 17s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.66199032
bestIteration = 538

Shrink model to first 539 iterations.

Training complete.
Selected model: catboost
    

In [10]:
rows = []
for strategy, result in results.items():
    for model, metrics in result["validation_metrics"].items():
        rows.append({
            "strategy": strategy,
            "model": model,
            **metrics,
            "effective_train_rows": result["weighting_summary"]["effective_train_rows"],
            "effective_sample_size": result["weighting_summary"]["effective_sample_size"],
            "mean_weight": result["weighting_summary"]["weight_mean"],
        })

comparison = pd.DataFrame(rows).sort_values(
    ["log_loss", "brier_score", "roc_auc"],
    ascending=[True, True, False],
)
comparison

,strategy,model,accuracy,balanced_accuracy,roc_auc,log_loss,brier_score,threshold,tn,fp,fn,tp,effective_train_rows,effective_sample_size,mean_weight
1,none,catboost,0.593069,0.594395,0.639753,0.661990,0.234977,0.5,293,231,180,306,6307,6307.000000,1.000000
0,none,random_forest,0.612871,0.614076,0.646797,0.662859,0.235200,0.5,305,219,172,314,6307,6307.000000,1.000000
6,exponential_half_life_8,random_forest,0.604950,0.605622,0.641418,0.663668,0.235625,0.5,308,216,183,303,6307,5423.037173,0.571970
5,step_before_2015_025,catboost,0.592079,0.594112,0.634848,0.663841,0.235862,0.5,283,241,171,315,6307,4710.999893,0.643491
4,step_before_2015_025,random_forest,0.601980,0.602311,0.638909,0.664350,0.235961,0.5,311,213,189,297,6307,4710.999893,0.643491
7,exponential_half_life_8,catboost,0.600990,0.602775,0.635276,0.664476,0.236062,0.5,291,233,170,316,6307,5423.037173,0.571970
3,exclude_before_2015,catboost,0.593069,0.594022,0.632598,0.666309,0.236893,0.5,298,226,185,301,3309,3309.000000,1.000000
2,exclude_before_2015,random_forest,0.604950,0.604950,0.629720,0.667439,0.237416,0.5,317,207,192,294,3309,3309.000000,1.000000


## 3. Initial Findings

CatBoost improves slightly when a limited amount of older data is removed, but performance worsens with the 2015 cutoff. Exponential decay provides no benefit, while step weighting is nearly equivalent to a hard cutoff. Random Forest does not improve under these weighting strategies.


## 4. Refined Cutoff Search

Several cutoff years between 2000 and 2010 are compared to determine whether a stable boundary exists.


In [11]:
rf_config = RandomForestConfig()
cat_config = CatBoostConfig()
split_config = SplitConfig()

strategies = {
    "none": TimeWeightConfig(mode="none"),
    "exclude_before_2000": TimeWeightConfig(
        mode="exclude_before", cutoff_date="2000-01-01"
    ),
    "exclude_before_2002": TimeWeightConfig(
            mode="exclude_before", cutoff_date="2002-01-01"
    ),
    "exclude_before_2004": TimeWeightConfig(
            mode="exclude_before", cutoff_date="2004-01-01"
    ),
    "exclude_before_2006": TimeWeightConfig(
            mode="exclude_before", cutoff_date="2006-01-01"
    ),
    "exclude_before_2008": TimeWeightConfig(
            mode="exclude_before", cutoff_date="2008-01-01"
    ),
    "exclude_before_2010": TimeWeightConfig(
            mode="exclude_before", cutoff_date="2010-01-01"
    ),
}

In [12]:
results = {}
for name, config in strategies.items():
    print("\n" + "=" * 80)
    print(name)
    results[name] = train_time_weighted_experiment(
        project_root=project_root,
        time_weight_config=config,
        rf_config=rf_config,
        cat_config=cat_config,
        split_config=split_config,
        evaluate_test=False,
        experiment_name=name,
    )


none
Temporal weighting:
  mode=none; original train=6,307; effective train=6,307; mean weight=1.0000; effective sample size=6307.0
Training Random Forest...
Training CatBoost...
0:	learn: 0.6919777	test: 0.6921598	best: 0.6921598 (0)	total: 44.7ms	remaining: 1m 29s
100:	learn: 0.6436785	test: 0.6693868	best: 0.6693868 (100)	total: 5.69s	remaining: 1m 47s
200:	learn: 0.6269106	test: 0.6643399	best: 0.6643151 (198)	total: 11.3s	remaining: 1m 41s
300:	learn: 0.6131275	test: 0.6635690	best: 0.6632978 (288)	total: 16.8s	remaining: 1m 34s
400:	learn: 0.5961288	test: 0.6627661	best: 0.6627661 (400)	total: 22.4s	remaining: 1m 29s
500:	learn: 0.5796968	test: 0.6623928	best: 0.6620925 (493)	total: 27.8s	remaining: 1m 23s
600:	learn: 0.5633403	test: 0.6636993	best: 0.6619903 (538)	total: 33.3s	remaining: 1m 17s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.66199032
bestIteration = 538

Shrink model to first 539 iterations.

Training complete.
Selected model: catboost
    

In [13]:
rows = []
for strategy, result in results.items():
    for model, metrics in result["validation_metrics"].items():
        rows.append({
            "strategy": strategy,
            "model": model,
            **metrics,
            "effective_train_rows": result["weighting_summary"]["effective_train_rows"],
            "effective_sample_size": result["weighting_summary"]["effective_sample_size"],
            "mean_weight": result["weighting_summary"]["weight_mean"],
        })

comparison = pd.DataFrame(rows).sort_values(
    ["log_loss", "brier_score", "roc_auc"],
    ascending=[True, True, False],
)
comparison

,strategy,model,accuracy,balanced_accuracy,roc_auc,log_loss,brier_score,threshold,tn,fp,fn,tp,effective_train_rows,effective_sample_size,mean_weight
3,exclude_before_2000,catboost,0.602970,0.604385,0.644645,0.660568,0.234221,0.5,297,227,174,312,6088,6088.0,1.0
13,exclude_before_2010,catboost,0.599010,0.600642,0.646208,0.661049,0.234352,0.5,292,232,173,313,5057,5057.0,1.0
9,exclude_before_2006,catboost,0.602970,0.604758,0.641371,0.661885,0.234841,0.5,292,232,169,317,5798,5798.0,1.0
1,none,catboost,0.593069,0.594395,0.639753,0.661990,0.234977,0.5,293,231,180,306,6307,6307.0,1.0
5,exclude_before_2002,catboost,0.593069,0.594395,0.642941,0.662030,0.234832,0.5,293,231,180,306,6007,6007.0,1.0
11,exclude_before_2008,catboost,0.603960,0.605488,0.638763,0.662368,0.235127,0.5,296,228,172,314,5471,5471.0,1.0
10,exclude_before_2008,random_forest,0.605941,0.606725,0.646526,0.662513,0.235057,0.5,307,217,181,305,5471,5471.0,1.0
7,exclude_before_2004,catboost,0.600000,0.600925,0.640000,0.662818,0.235239,0.5,302,222,182,304,5916,5916.0,1.0
12,exclude_before_2010,random_forest,0.607921,0.609006,0.646165,0.662823,0.235191,0.5,304,220,176,310,5057,5057.0,1.0
0,none,random_forest,0.612871,0.614076,0.646797,0.662859,0.235200,0.5,305,219,172,314,6307,6307.0,1.0


## Summary

Progressive weighting was discarded. Removing some early fights produced a small CatBoost improvement, but no cutoff was consistent enough to include in the final training strategy.
